# Fair DFL Experiment Runner

Minimal notebook for running experiments on Colab.  
All logic lives in `run_methods.py` (CLI) and `configs.py` (shared config).  
For analysis and plotting, see `notebook_analysis.ipynb`.

In [ ]:
# Install dependencies
!pip install -q torch numpy pandas scipy matplotlib cvxpy mosek
!pip install -q git+https://github.com/cvxpy/cvxtorch.git
!pip install -q --upgrade threadpoolctl

In [ ]:
import os
import sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ----- Path setup -----
DRIVE_ROOT = '/content/drive/MyDrive/fdfl_experiment'
MOSEK_LIC_PATH = f'{DRIVE_ROOT}/mosek.lic'

# Set MOSEK license
if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH
    print(f'MOSEK license set: {MOSEK_LIC_PATH}')
else:
    print(f'Warning: MOSEK license not found at {MOSEK_LIC_PATH}')

# Add both DRIVE_ROOT (for configs.py) and src/ (for fair_dfl package)
for p in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Change to project directory so run_methods.py can find configs.py
%cd {DRIVE_ROOT}
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Verify imports — unified training pipeline
from fair_dfl.runner import run_experiment_unified
from configs import ALL_METHOD_CONFIGS, describe_method

print(f'{len(ALL_METHOD_CONFIGS)} methods available:')
for name, spec in ALL_METHOD_CONFIGS.items():
    print(f'  {name:15s} -> {describe_method(name, spec)}')

## Run Experiments

Use `run_methods.py` CLI to run any subset of methods.  
Results are appended to `results/stage_results_full.csv` (deduplicates automatically).

```
Common flags:
  --methods NAME1 NAME2   Run specific methods
  --all                   Run all methods
  --alphas 0.5 2.0        Alpha-fairness values
  --n-sample 500          Quick test (0 = all data)
  --dry-run               Preview without running
  --overwrite             Start fresh instead of appending
  --list-methods          Show all available methods
```

In [ ]:
# List all available methods
!python run_methods.py --list-methods

In [ ]:
# Quick test with 500 samples (verify everything works)
!python run_methods.py --methods FPTO FDFL PCGrad --n-sample 500 --alphas 0.5 --overwrite

In [ ]:
# Full experiment — run all core methods on all data
# Uncomment and edit as needed:

# !python run_methods.py --methods FPTO FDFL FFO WS-equal WS-dec MGDA PCGrad --alphas 0.5 2.0 --overwrite

In [ ]:
# Add more methods without rerunning existing ones (append mode)
# !python run_methods.py --methods PLG-FP PLG-PP --alphas 0.5 2.0

In [ ]:
# Check what results we have
!ls -la results/